# Visualize Elastic Disks Data

Load one generated sequence and display it as an animated GIF.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/jordanshivers/latent-video-forecasting.git"
REPO_DIRNAME = "latent-video-forecasting"
INSTALL_REQUIREMENTS = True

_current = Path.cwd().resolve()
REPO_ROOT = None
for _candidate in [_current, *_current.parents]:
    if (_candidate / "requirements.txt").exists() and (_candidate / "src" / "video_forecasting").is_dir():
        REPO_ROOT = _candidate
        break

if REPO_ROOT is None:
    _in_colab = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
    if _in_colab:
        _target = Path("/content") / REPO_DIRNAME
        if not _target.exists():
            subprocess.check_call(["git", "clone", GITHUB_REPO_URL, str(_target)])
        REPO_ROOT = _target
    else:
        raise FileNotFoundError(f"Could not find repo root from {_current}.")

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

if INSTALL_REQUIREMENTS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")])

from video_forecasting.runtime import get_data_dir, get_output_dir

DATA_DIR = get_data_dir(REPO_ROOT)
OUTPUT_DIR = get_output_dir("visualize_elastic_disks_data", REPO_ROOT)
print(f"Repo root: {REPO_ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"Output dir: {OUTPUT_DIR}")


In [ ]:
import imageio.v2 as imageio
import numpy as np
from IPython.display import Image, display

from video_forecasting.datasets.elastic_disks import ElasticDisksDataset


In [ ]:
dataset = ElasticDisksDataset(root=str(DATA_DIR), train=True, num_sequences=40, sequence_length=50, num_particles=10, boundary="periodic", normalize=True, frame_separation=1, seed=42, max_sequences=20, render_mode='hard')
sequence = dataset.sequences[0]
frames = []
for frame in sequence:
    gray = np.clip(frame[0], 0, 1)
    rgb = np.stack([gray, gray, gray], axis=2)
    frames.append((rgb * 255).astype(np.uint8))

gif_path = OUTPUT_DIR / "elastic_disks_sequence.gif"
imageio.mimsave(gif_path, frames, fps=8, loop=0)
print(f"Saved GIF to: {gif_path}")
display(Image(filename=str(gif_path)))
